# Sampling layer elevations and handling pinch-outs

Build a Voronoi plan mesh, sample a dipping surface at its generators and inspect a lens that pinches out. The last cell attaches the DISU package to a FloPy groundwater model and plots where the lens is active. All steps run without a MODFLOW executable.

In [ ]:
import numpy as np
from shapely.geometry import box
from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu
from vorflow.utils import grid_to_disv
import flopy
from flopy.discretization import VertexGrid

cm = ConceptualMesh()
cm.add_polygon(box(0, 0, 100, 100), zone_id=1, resolution=15)
mg = MeshGenerator(background_lc=15, conceptual_mesh=cm, verbosity=0)
mg.generate()
grid = VoronoiTessellator(mg, cm).generate()

## Make the sampling grid

The DISV geometry export contains the vertices and the generator coordinates in each `cell2d` record. These define a FloPy plan grid before the elevations are known. The same plan grid serves every layer in this example.

In [ ]:
def plan_grid(grid):
    p = grid_to_disv(grid)
    return VertexGrid(
        vertices=p["vertices"], cell2d=p["cell2d"],
        ncpl=p["ncpl"], nlay=1, crs=grid.crs,
    )

plan = plan_grid(grid)

## Set the elevations

The model top dips east at 3%. Evaluate it directly, or set `use_raster = True` to sample the same surface from a small in-memory FloPy raster. The raster option uses Rasterio and Rasterstats. Pixel centres extend beyond the model boundary so linear interpolation covers the entire mesh. With a DEM file, use `Raster.load(path)` and supply the mesh and raster in the same coordinate system.

In [ ]:
use_raster = False
x = plan.xcellcenters
if use_raster:
    from flopy.utils import Raster
    from rasterio.transform import from_origin

    pixel_x = -5.0 + 10.0 * np.arange(12)
    dem_values = np.tile(100.0 - 0.03 * pixel_x, (12, 1))
    dem = Raster(dem_values[None, :, :], bands=(1,), crs=None,
                 transform=from_origin(-10, 110, 10, 10), nodataval=-9999.0)
    top = dem.resample_to_grid(plan, band=1, method="linear")
else:
    top = 100.0 - 0.03 * x

lens_top = top - 5.0
lens_thickness = np.maximum(0.0, 2.0 - x / 30.0)
bottoms = [lens_top, lens_top - lens_thickness, top - 20.0]
botm = np.stack(bottoms)

## Choose the cell status

Here the first and third layers stay active. The modeller excludes part of the middle layer from flow. For a DISV package, mark any remaining thin cells as vertical pass-through with `-1`. The surface above each layer is the previous layer’s bottom. Resolve missing values and crossed horizons before applying this rule.

In [ ]:
min_thickness = 0.1  # metres
domain = np.ones(botm.shape, dtype=int)
domain[1, plan.ycellcenters < 15.0] = 0

disv_idomain = domain.copy()
layer_top = top
for k, layer_bottom in enumerate(botm):
    thickness = layer_top - layer_bottom
    thin_active = (domain[k] > 0) & (thickness <= min_thickness)
    disv_idomain[k, thin_active] = -1
    layer_top = layer_bottom

## Build and view DISU

Pass the original domain mask and threshold to Vorflow. The builder removes pinched cells, connects active nodes above and below them, and honours explicit no-flow cells. Its default collapse rule shares a removed interval between adjacent active cells. For DISV the pass-through flag leaves the supplied elevations in place.

In [ ]:
d = disu.DisuGrid(grid, top, botm, domain, min_thickness=min_thickness).build()
sim = flopy.mf6.MFSimulation()
gwf = flopy.mf6.ModflowGwf(sim, modelname="lens")
package = flopy.mf6.ModflowGwfdisu(gwf, **d.get_gridprops_mf6_disu())

view = flopy.plot.PlotMapView(modelgrid=plan)
view.plot_array(np.ma.masked_where(d.nodemap[1] < 0, lens_thickness))
view.plot_grid(linewidth=0.3)
view.ax.set_title("Active lens thickness (m)")
print(d.report())